### 傳統 ML 模型訓練與比較

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')  # 添加父目錄到Python路徑
from src.evaluate import evaluate_model, evaluate_threshold, plot_feature_importance_xgb
from src.visualization.plotting import visualize_threshold_results

df = pd.read_csv('../data/cleaned_encoded_data.csv')
print(df.appointment_shift.head(5))
df.isna().sum()
df.info()

In [ ]:
df.info()

In [ ]:
# Split Features and Target
X = df.drop(columns=['no_show', 'icd'])
y = df['no_show']


In [ ]:
# Train-Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42) # Stratify to maintain the distribution of the target variable in both train and test sets


### Model Training 
#### 1. Logistic Regression
#### 2. SVM
#### 3. Random Forest 
#### 4. XGboost 

In [ ]:

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
# import standard scaler
from sklearn.preprocessing import StandardScaler


In [ ]:
# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
### Add Imputer and Missing indicators for missing values
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import StandardScaler

# 首先創建缺失值指示器
missing_indicator = X_train['age'].isna().astype(int).values.reshape(-1, 1)
missing_indicator_test = X_test['age'].isna().astype(int).values.reshape(-1, 1)

# 創建和使用 imputer: missing value indicator + imputed features
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)


# 特徵縮放
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# 將缺失值指示器與縮放後的特徵組合
X_train_final = np.hstack((missing_indicator, X_train_scaled))
X_test_final = np.hstack((missing_indicator_test, X_test_scaled))

# Final Features: X_train_final, X_test_final (Imbalanced)


In [ ]:
# Fix Imbalance data using SMOTE: X_train_balanced, y_train_balanced
print("Before SMOTE:")
print(y_train.value_counts())

from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_final, y_train)

print("After SMOTE:")
print(y_train_balanced.value_counts())

#### 1. X_train_final: Original imbalanced data with imputation and missing indicators, pair with Random Forest using class_weight#
#### 2. X_train_balanced: Balanced data using SMOTE: includes synthetic data, might not reflect true conditions

In [ ]:
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_balanced, y_train_balanced)

In [ ]:
y_pred_lg = logreg.predict(X_test_final)
y_proba_lg = logreg.predict_proba(X_test_final)[:, 1]

#### Evaluation Metrics:
1. Accuracy = (TP + TN) / (TP + FP + TN + FN) 
2. Precision = TP / (TP + FN) 
3. Recall = TP / (TP + EN) -> 成功預測出來的比例
4. F1 Score = 2 * (Precision + Recall) / (Precision + Recall) 
5. ROC AUC

In [ ]:
evaluate_model(y_test, y_pred_lg, y_proba_lg, "Logistic Regression")

In [ ]:
## Adjust threshold to improve recall
results_df = evaluate_threshold(y_test, y_proba_lg)
visualize_threshold_results(results_df)


## SVM

In [ ]:
from sklearn.svm import SVC

# 2. 定義和訓練 SVM 模型
svm_model = SVC(
    kernel='rbf',  # 使用 RBF 核函數
    probability=True,  # 需要這個來獲得概率預測
    class_weight='balanced',  # 處理類別不平衡
    random_state=42
)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score
# 3. 使用 GridSearchCV 尋找最佳參數
param_grid_svm = {
    'C': [1],           # 先只用默認值
    'gamma': ['scale'], # 先只用默認值
    'class_weight': ['balanced', {0:1, 1:10}]  # 只測試兩種權重
}

# 2. 減少交叉驗證折數
grid_svm = GridSearchCV(
    svm_model,
    param_grid_svm,
    scoring={
        'recall': make_scorer(recall_score, pos_label=1),
        'roc_auc': 'roc_auc'
    },
    refit='recall',
    cv=3,  # 從5減到3
    n_jobs=-1,
    verbose=2
)

# 4. 訓練模型
print("Training SVM model...")
grid_svm.fit(X_train_scaled, y_train)




In [ ]:
# 5. 獲取最佳模型
best_svm = grid_svm.best_estimator_

# 6. 進行預測
y_pred_svm = best_svm.predict(X_test_scaled)
y_proba_svm = best_svm.predict_proba(X_test_scaled)[:, 1]

# 7. 評估模型
print("\nSVM Model Evaluation:")
print("Best parameters:", grid_svm.best_params_)

evaluate_model(y_test, y_pred_svm, y_proba_svm, "SVM")
results_df = evaluate_threshold(y_test, y_proba_svm)
visualize_threshold_results(results_df)

## Random Forest 

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,        # number of trees
    max_depth=None,          # allow trees to grow fully
    random_state=42,
    class_weight=None        # SMOTE already handled imbalance
)

rf_model.fit(X_train_balanced, y_train_balanced)


In [ ]:
y_pred_rf = rf_model.predict(X_test_final)
y_proba_rf = rf_model.predict_proba(X_test_final)[:, 1]  # needed for AUC

In [ ]:
## Initial Model
evaluate_model(y_test, y_pred_rf, y_proba_rf, "Random Forest")


In [ ]:
## Weighted models (Dealing with the Imbalanced Dataset)
rf_model_2 = RandomForestClassifier(
    n_estimators=200,        # number of trees
    max_depth=None,          # allow trees to grow fully
    random_state=42,
    class_weight='balanced',        
    min_samples_split=2
)

rf_model_2.fit(X_train_final, y_train)
y_pred_rf_weighted = rf_model_2.predict(X_test_final)
y_proba_rf_weighted = rf_model_2.predict_proba(X_test_final)[:, 1]  # needed for AUC

In [ ]:
evaluate_model(y_test, y_pred_rf_weighted, y_proba_rf_weighted, "Random Forest_Weighted")

# Determine the best threshold for cost saving
results_df = evaluate_threshold(y_test, y_proba_rf_weighted)
visualize_threshold_results(results_df)


In [ ]:
threshold = 0.1
y_pred_custom = (y_proba_rf_weighted >= threshold).astype(int)
evaluate_model(y_test, y_pred_custom, y_proba_rf_weighted, "Random Forest_Weighted_Custom_Threshold_0.1")


In [ ]:
# 1. Grid Search Paramters
param_grid_rf_fine = {
    'n_estimators': [180, 200, 220],  # 圍繞200探索
    'max_depth': [None, 30, 40],      # 測試是否需要限制深度
    'min_samples_split': [2, 3],      # 微調分割條件
    'min_samples_leaf': [1, 2],       # 控制葉節點大小
    'max_features': ['sqrt', 'log2'],  # 特徵選擇方法
    'class_weight': [
        'balanced',
        {0:1, 1:8},
        {0:1, 1:10},
        {0:1, 1:12}
    ]  # 更細緻的類別權重
}

# Define Cost Saving to used as Scoring 
def calculate_cost_savings(cm_values):
    """計算成本節省"""
    tn, fp, fn, tp = cm_values
    savings = tp * 200  # 每個正確預測的no-show節省200
    costs = (fp * 20)   # 每個誤報成本20
    missed = fn * 200   # 每個漏報成本200
    return savings - costs - missed

# 2. 使用多個評估指標的 GridSearchCV
grid_rf_fine = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf_fine,
    scoring={
        'recall': make_scorer(recall_score, pos_label=1),  # no-show的召回率
        'roc_auc': 'roc_auc',                             # 整體表現
        'cost_savings': make_scorer(
            lambda y_true, y_pred: calculate_cost_savings(
                confusion_matrix(y_true, y_pred).ravel()
            )
        )  # 自定義成本評分
    },
    refit='cost_savings',  # 使用成本節省作為選擇標準
    cv=5,
    n_jobs=-1,
    verbose=2
)

print("開始微調...")
grid_rf_fine.fit(X_train, y_train)

In [ ]:

# 5. 獲取最佳模型
best_rf = grid_rf_fine.best_estimator_
print("\n最佳參數:", grid_rf_fine.best_params_)

# 6. 評估最佳模型
y_pred_rf_weighted_best = best_rf.predict(X_test)
y_proba_rf_weighted_best = best_rf.predict_proba(X_test)[:, 1]

# 7. 詳細評估
print("\n最佳模型評估:")
evaluate_model(y_test, y_pred_rf_weighted_best, y_proba_rf_weighted_best, "RF_Fine_Tuned")

# 8. 閾值優化
results_df = evaluate_threshold(y_test, y_proba_rf_weighted_best)
print("\n閾值優化結果:")
print(results_df.sort_values('cost_savings', ascending=False).head())

visualize_threshold_results(results_df)

# 9. 特徵重要性分析
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\n最重要的特徵:")
print(feature_importance.head(10))



In [ ]:
threshold = 0.25
y_pred_custom = (y_proba_rf_weighted_best >= threshold).astype(int)
evaluate_model(y_test, y_pred_custom, y_proba_rf_weighted_best, "RF_Fine_Tuned_Custom_Threshold_0.25")


## No differences between weighted and SMOTE
1. SMOTE creates synthetic examples that may not generalize well

2. Random Forest already does internal bootstrapping + averaging

3. Giving the model correct penalties via class weights is more elegant and stable



In [ ]:
grid_rf_fine.fit(X_train_final, y_train)

In [ ]:
# 5. 獲取最佳模型
best_rf = grid_rf_fine.best_estimator_
print("\n最佳參數:", grid_rf_fine.best_params_)

# 6. 評估最佳模型
y_pred_rf_weighted_best = best_rf.predict(X_test_final)
y_proba_rf_weighted_best = best_rf.predict_proba(X_test_final)[:, 1]

# 7. 詳細評估
print("\n最佳模型評估:")
evaluate_model(y_test, y_pred_rf_weighted_best, y_proba_rf_weighted_best, "RF_Fine_Tuned")

# 8. 閾值優化
results_df = evaluate_threshold(y_test, y_proba_rf_weighted_best)
print("\n閾值優化結果:")
print(results_df.sort_values('cost_savings', ascending=False).head())

visualize_threshold_results(results_df)

In [ ]:
threshold = 0.30
y_pred_custom = (y_proba_rf_weighted_best >= threshold).astype(int)
evaluate_model(y_test, y_pred_custom, y_proba_rf_weighted_best, "RF_Fine_Tuned_Custom_Threshold_0.30")

In [ ]:
grid_rf_fine.fit(X_train_balanced, y_train_balanced)

In [ ]:
# 5. 獲取最佳模型
best_rf_balanced = grid_rf_fine.best_estimator_
print("\n最佳參數:", grid_rf_fine.best_params_)

# 6. 評估最佳模型
y_pred_rf_balanced_best = best_rf_balanced.predict(X_test_final)
y_proba_rf_balanced_best = best_rf_balanced.predict_proba(X_test_final)[:, 1]

# 7. 詳細評估
print("\n最佳模型評估:")
evaluate_model(y_test, y_pred_rf_balanced_best, y_proba_rf_balanced_best, "RF_Fine_Tuned_Balanced")

# 8. 閾值優化
results_df = evaluate_threshold(y_test, y_proba_rf_balanced_best)
print("\n閾值優化結果:")
print(results_df.sort_values('cost_savings', ascending=False).head())

visualize_threshold_results(results_df)

### XGBoost

In [ ]:
import xgboost as xgb

In [ ]:
# Compute ratio: (negative class / positive class) for imbalanced data
# So that the model pays more attention to the minority class
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", ratio)


In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=ratio,    
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train, y_train)


In [ ]:
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

evaluate_model(y_test, y_pred_xgb, y_proba_xgb, "XGBoost")


In [ ]:
threshold = 0.35
y_pred_custom = (y_proba_xgb >= threshold).astype(int)
print("Custom Threshold Report:")
print(classification_report(y_test, y_pred_custom))


## For Shows (Class 0):
1. Precision: 0.97 → 97% of patients predicted to show up actually showed up (improved)
2. Recall: 0.44 → Only 44% of actual show-ups were correctly identified (decreased)
3. F1-score: 0.61 → Moderate balanced performance for show-ups

## For No-shows (Class 1):
1. Precision: 0.15 → Only 15% of patients predicted to no-show actually didn't show up
2. Recall: 0.86 → 86% of actual no-shows were correctly identified (significantly improved)
3. F1-score: 0.25 → Still weak overall performance, but better at catching no-shows

In [ ]:
## Fine Tuning XGBoost Model
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3],              # Add to prevent overfitting                    
    'reg_alpha': [0, 0.1, 1.0],                 # L1 regularization
    'reg_lambda': [0, 1.0, 5.0]                 # L2 regularization
}

xgb_clf = xgb.XGBClassifier(
    scale_pos_weight=ratio,
    eval_metric='logloss',
    random_state=42
)

# grid_search = GridSearchCV(
#     estimator=xgb_clf,
#     param_grid=param_grid,
#     scoring='recall',         # optimize for recall!
#     cv=3,
#     n_jobs=-1,
#     verbose=2
# )

from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=xgb_clf,
    param_distributions=param_grid,
    n_iter=200,             
    scoring='recall',
    cv=3,
    n_jobs=-1,
    verbose=2
)





In [ ]:
# grid_search.fit(X_train, y_train)
random_search.fit(X_train, y_train)

In [ ]:
best_xgb = random_search.best_estimator_

print("Best parameters:", random_search.best_params_)

# Predict and evaluate
y_pred_xgb_best = best_xgb.predict(X_test)
y_proba_xgb_best = best_xgb.predict_proba(X_test)[:, 1]

evaluate_model(y_test, y_pred_xgb_best, y_proba_xgb_best, "XGBoost_Best")
results_df = evaluate_threshold(y_test, y_proba_xgb_best)
visualize_threshold_results(results_df)


In [ ]:
threshold = 0.45
y_pred_thresholded = (y_proba_xgb_best >= threshold).astype(int)

print("Threshold-Tuned Report:")
print(classification_report(y_test, y_pred_thresholded))


In [ ]:
# # Feature Importance

plot_feature_importance_xgb(best_xgb)

In [ ]:
# Top features based on feature importance score
# Select top 15 features
importances = best_xgb.get_booster().get_score(importance_type='weight')
top_features = sorted(importances, key=importances.get, reverse=True)[:15]
print(top_features)

# Create a dataset with only top features
X_train_top = X_train[top_features]
X_test_top = X_test[top_features]


# Train model with just these features
xgb_top = xgb.XGBClassifier(
    scale_pos_weight=ratio,
    eval_metric='logloss',
    random_state=42
)

xgb_top.fit(X_train_top, y_train)

# Compare performance with full-feature model
y_pred_top_xgb = xgb_top.predict(X_test_top)
print(classification_report(y_test, y_pred_top_xgb))
y_proba_top_xgb = xgb_top.predict_proba(X_test_top)[:, 1]

evaluate_model(y_test, y_pred_top_xgb, y_proba_top_xgb, "XGBoost_Top_Features")
results_df = evaluate_threshold(y_test, y_proba_top_xgb)
visualize_threshold_results(results_df)


In [ ]:
threshold = 0.45
y_pred_thresholded = (y_proba_top_xgb >= threshold).astype(int)

print("Threshold-Tuned Report:")
print(classification_report(y_test, y_pred_thresholded))


In [ ]:
# Further Enhance Model Performance by creating interactions between features
# Create interactions between top features
X_train_enhanced = X_train[top_features].copy()
X_test_enhanced = X_test[top_features].copy()

# Weather x Age interactions
X_train_enhanced['temp_age_interaction'] = X_train['max_temp_day'] * X_train['age']
X_test_enhanced['temp_age_interaction'] = X_test['max_temp_day'] * X_test['age']

X_train_enhanced['rain_age_interaction'] = X_train['average_rain_day'] * X_train['age']
X_test_enhanced['rain_age_interaction'] = X_test['average_rain_day'] * X_test['age']

# Month x Weather interactions
for month in ['appointment_month_6', 'appointment_month_7', 'appointment_month_8']:
    X_train_enhanced[f'{month}_temp'] = X_train[month] * X_train['max_temp_day']
    X_test_enhanced[f'{month}_temp'] = X_test[month] * X_test['max_temp_day']
    
    X_train_enhanced[f'{month}_rain'] = X_train[month] * X_train['average_rain_day']
    X_test_enhanced[f'{month}_rain'] = X_test[month] * X_test['average_rain_day']



In [ ]:
# Focus parameter grid on feature-related parameters
focused_param_grid = {
    'n_estimators': [200, 300],          # Reduced options
    'max_depth': [4, 6],                 # Reduced options
    'learning_rate': [0.05, 0.1],        # Smaller learning rates
    'colsample_bytree': [0.6, 0.7, 0.8], # Focus on feature sampling
    'subsample': [0.8, 0.9],             # Less row sampling variation
    'gamma': [0.1, 0.3, 0.5],            # More focus on feature split quality
    'min_child_weight': [1, 3]           # Fewer options
}

# Create model for grid search
xgb_focused = xgb.XGBClassifier(
    scale_pos_weight=ratio,
    eval_metric='logloss',
    random_state=42
)

# Run grid search
from sklearn.model_selection import StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid search with custom metric focusing on recall
from sklearn.metrics import make_scorer, fbeta_score
custom_scorer = make_scorer(fbeta_score, beta=2)

grid_search_focused = GridSearchCV(
    estimator=xgb_focused,
    param_grid=focused_param_grid,
    scoring=custom_scorer,
    cv=cv,
    n_jobs=-1,
    verbose=2
)

# Fit on enhanced features dataset
grid_search_focused.fit(X_train_enhanced, y_train)

In [ ]:
# Get best model from grid search
best_model = grid_search_focused.best_estimator_

print("Best parameters:", grid_search_focused.best_params_)

# Predict and evaluate
y_pred_best_enhanced = best_model.predict(X_test_enhanced)
y_proba_best_enhanced = best_model.predict_proba(X_test_enhanced)[:, 1]

evaluate_model(y_test, y_pred_best_enhanced, y_proba_best_enhanced, "XGBoost_Best")
results_df = evaluate_threshold(y_test, y_proba_best_enhanced)
visualize_threshold_results(results_df)



In [ ]:
threshold = 0.45
y_pred_thresholded = (y_proba_best_enhanced >= threshold).astype(int)

print("Threshold-Tuned Report:")
print(classification_report(y_test, y_pred_thresholded))



In [ ]:
def optimize_and_compare_models(models_dict):
    """
    為每個模型找出最佳閾值並比較結果
    
    models_dict = {
        'model_name': {
            'y_proba': probabilities,
            'y_test': true_labels
        }
    }
    """
    comparison_results = []
    
    for model_name, data in models_dict.items():
        # 使用evaluate_threshold找出最佳閾值
        results_df = evaluate_threshold(data['y_test'], data['y_proba'])
        
        # 找出最佳成本節省的閾值
        best_idx = results_df['cost_savings'].idxmax()
        best_result = results_df.loc[best_idx]
        
        # 使用最佳閾值的預測結果
        y_pred_best = (data['y_proba'] >= best_result['threshold']).astype(int)
        tn, fp, fn, tp = confusion_matrix(data['y_test'], y_pred_best).ravel()
        
        # 計算詳細成本
        no_show_cost = 200  # 每個未檢測到的no-show成本
        intervention_cost = 20  # 每次干預成本
        
        total_savings = (tp * no_show_cost) - ((fp + tp) * intervention_cost)
        missed_cost = fn * no_show_cost
        intervention_total_cost = (fp + tp) * intervention_cost
        
        comparison_results.append({
            'Model': model_name,
            'Best Threshold': best_result['threshold'],
            'Recall (No-show)': tp / (tp + fn),
            'Precision': tp / (tp + fp) if (tp + fp) > 0 else 0,
            'True Positives': tp,
            'False Positives': fp,
            'False Negatives': fn,
            'Cost Savings ($)': total_savings,
            'Missed No-show Cost ($)': missed_cost,
            'Intervention Cost ($)': intervention_total_cost
        })
    
    # 創建比較表格
    comparison_df = pd.DataFrame(comparison_results)
    
    # 格式化數值
    # format_dict = {
    #     'Best Threshold': '{:.3f}',
    #     'Recall (No-show)': '{:.3f}',
    #     'Precision': '{:.3f}',
    #     'Cost Savings ($)': '${:,.0f}',
    #     'Missed No-show Cost ($)': '${:,.0f}',
    #     'Intervention Cost ($)': '${:,.0f}'
    # }
    
    # for col, format_str in format_dict.items():
    #     comparison_df[col] = comparison_df[col].map(format_str.format)
    
    return comparison_df

# 準備模型結果
models_dict = {
    'Logistic Regression': {
        'y_proba': y_proba_lg,
        'y_test': y_test
    },
    'SVM': {
        'y_proba': y_proba_svm,
        'y_test': y_test
    },
    'Random Forest (Original)': {
        'y_proba': y_proba_rf_weighted_best,
        'y_test': y_test
    },
    
    'Random Forest (SMOTE__Balanced)': {
        'y_proba': y_proba_rf_balanced_best,
        'y_test': y_test
    },
    'XGBoost': {
        'y_proba': y_proba_xgb,
        'y_test': y_test
    },
    'XGBoost (Best)': {
        'y_proba': y_proba_xgb_best,
        'y_test': y_test
    },
    'XGBoost (Top Features)': {
        'y_proba': y_proba_top_xgb,
        'y_test': y_test
    },
    'XGBoost (Enhanced)': {
        'y_proba': y_proba_best_enhanced,
        'y_test': y_test
    }
}

# 生成比較表    
comparison_df = optimize_and_compare_models(models_dict)

# 顯示結果
print("\nModel Comparison with Optimized Thresholds:")
print(comparison_df.sort_values('Cost Savings ($)', ascending=False))

# 計算總節省成本
total_savings = comparison_df['Cost Savings ($)'].max()
print(f"\nMaximum Potential Cost Savings: ${total_savings:,.2f}")

import matplotlib.pyplot as plt
# 視覺化比較
plt.figure(figsize=(15, 6))

# 成本節省比較
plt.subplot(1, 2, 1)
costs = comparison_df['Cost Savings ($)']
plt.bar(comparison_df['Model'], costs)
plt.title('Cost Savings Comparison')
plt.xticks(rotation=45)
plt.ylabel('Cost Savings ($)')

# 閾值和召回率比較
plt.subplot(1, 2, 2)
plt.scatter(comparison_df['Best Threshold'].astype(float), 
           comparison_df['Recall (No-show)'].astype(float))
plt.title('Threshold vs Recall')
plt.xlabel('Best Threshold')
plt.ylabel('Recall Score')
for i, model in enumerate(comparison_df['Model']):
    plt.annotate(model, 
                (comparison_df['Best Threshold'].astype(float).iloc[i],
                 comparison_df['Recall (No-show)'].astype(float).iloc[i]))

plt.tight_layout()
plt.show()

In [ ]:
print(comparison_df)
print(comparison_df.keys())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

colors = plt.cm.tab10(np.linspace(0, 1, len(comparison_df)))

# Convert necessary columns to float if not already
comparison_df['Cost Savings ($)'] = comparison_df['Cost Savings ($)'].astype(float)
comparison_df['Best Threshold'] = comparison_df['Best Threshold'].astype(float)
comparison_df['Recall (No-show)'] = comparison_df['Recall (No-show)'].astype(float)
comparison_df['Precision'] = comparison_df['Precision'].astype(float)

# Plotting
fig, axs = plt.subplots(2, 2, figsize=(16, 10))

# 1. Cost Savings Comparison
axs[0, 0].barh(comparison_df['Model'], comparison_df['Cost Savings ($)'], color='#2E86C1')
axs[0, 0].set_title('Figure 7A. Cost Savings Comparison by Model', fontsize=15)
axs[0, 0].set_xlabel('Cost Savings ($)', fontsize=15)
axs[0, 0].invert_yaxis()  # Highest at top
axs[0, 0].grid(axis='x', linestyle='--', alpha=0.7)

# 2. Threshold vs Recall
axs[0, 1].scatter(comparison_df['Best Threshold'], comparison_df['Recall (No-show)'], color= colors,s=100)
axs[0, 1].set_xlim(0.2, 0.5)
axs[0, 1].set_ylim(0.60, 0.80)
for i, row in comparison_df.iterrows():
    axs[0, 1].annotate(row['Model'], (row['Best Threshold'], row['Recall (No-show)']), fontsize=15)
axs[0, 1].set_title('Figure 7B. Threshold vs Recall', fontsize=15)
axs[0, 1].set_xlabel('Best Threshold', fontsize=15)
axs[0, 1].set_ylabel('Recall (No-show)', fontsize=15)
axs[0, 1].grid(True, linestyle='--', alpha=0.7)

# 3. Recall vs Precision (PR positioning)
axs[1, 0].scatter(comparison_df['Precision'], comparison_df['Recall (No-show)'], s=100, color=colors)
axs[1, 0].set_xlim(0.1, 0.32)
axs[1, 0].set_ylim(0.62, 0.80)
for i, row in comparison_df.iterrows():
    axs[1, 0].annotate(row['Model'], (row['Precision'], row['Recall (No-show)']), fontsize=15)
axs[1, 0].set_title('Precision vs Recall')
axs[1, 0].set_xlabel('Precision', fontsize=15)
axs[1, 0].set_ylabel('Recall', fontsize=15)
axs[1, 0].grid(True, linestyle='--', alpha=0.7)

# 4. Threshold Histogram
axs[1, 1].hist(comparison_df['Best Threshold'], bins=np.linspace(0.1, 0.5, 9), color='teal', edgecolor='black')
axs[1, 1].set_title('Distribution of Best Thresholds', fontsize=15)
axs[1, 1].set_xlabel('Best Threshold', fontsize=15)
axs[1, 1].set_ylabel('Number of Models', fontsize=15)
axs[1, 1].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()
